# 一步步构建深度神经网络

欢迎来到第 4 周作业（第 1 部分，共 2 部分）！之前你使用单个隐藏层训练了 2 层神经网络。本周，你将构建一个具有任意层数的深度神经网络！

- 在此 Notebook 中，你将实现构建深度神经网络所需的所有函数。
- 对于下一个作业，你将使用这些函数构建用于图像分类的深度神经网络。

**在本作业结束时，你将能够：**

- 使用 ReLU 等非线性单元来改进模型
- 构建更深的神经网络（超过1个隐藏层）
- 实现一个易于使用的神经网络类

**符号**：
- 上标$[l]$表示与$l^{th}$层关联的数量。 
    - 示例：$a^{[L]}$ 是 $L^{th}$ 层激活值。 $W^{[L]}$ 和 $b^{[L]}$ 是 $L^{th}$ 层参数。
- 上标 $(i)$ 表示与 $i^{th}$ 示例关联的数量。 
    - 示例：$x^{(i)}$ 是 $i^{th}$ 训练样本。
- 小写$i$表示向量的$i^{th}$条目。
    - 示例：$a^{[l]}_i$ 表示 $l^{th}$ 层的激活值的 $i^{th}$ 条目。

让我们开始吧！

## 关于提交至 AutoGrader 的重要说明

向 AutoGrader 提交作业前，请确认满足以下要求：

1. 未在作业中添加 _额外的_  `print` 语句。
2. 未在作业中添加 _额外的_ 代码单元格。
3. 未修改任何函数的参数。
4. 未在计分练习中使用全局变量。除非明确要求使用全局变量，否则应使用局部变量。
5. 未修改无需改动的作业代码，例如创建 _额外的_ 变量。

如果违反上述要求，提交作业时可能出现 `Grader Error: Grader feedback not found` 或其他意外错误。在求助或调试作业之前，请先检查这些情况。如果不记得自己做过哪些改动，可以按照 [说明](https://www.coursera.org/learn/neural-networks-deep-learning/supplement/iLwon/h-ow-to-refresh-your-workspace) 获取一份新的作业。

## 目录
- [1 - 依赖库](#1)
- [2 - 概览](#2)
- [3 - 初始化](#3)
    - [3.1 - 2 层神经网络](#3-1)
        - [练习 1 - initialize_parameters](#ex-1)
    - [3.2 - L层神经网络](#3-2)
        - [练习 2 - initialize_parameters_deep](#ex-2)
- [4 - 前向传播模块](#4)
    - [4.1 - 线性部分的前向传播](#4-1)
        - [练习 3 - linear_forward](#ex-3)
    - [4.2 - 线性与激活部分的前向传播](#4-2)
        - [练习 4 - linear_activation_forward](#ex-4)
    - [4.3 - L层模型](#4-3)
        - [练习 5 - L_model_forward](#ex-5)
- [5 - 代价函数](#5)
    - [练习 6 - compute_cost](#ex-6)
- [6 - 反向传播模块](#6)
    - [6.1 - 线性部分的反向传播](#6-1)
        - [练习 7 - linear_backward](#ex-7)
    - [6.2 - 线性与激活部分的反向传播](#6-2)
        - [练习 8 - linear_activation_backward](#ex-8)
    - [6.3 - L 层模型的反向传播](#6-3)
        - [练习 9 - L_model_backward](#ex-9)
    - [6.4 - 更新参数](#6-4)
        - [练习 10 - update_parameters](#ex-10)

<a name='1'></a>
## 1 - 依赖库

首先，导入在此作业期间需要的所有包。 

- [NumPy](www.numpy.org) 是使用 Python 进行科学计算的主要包。
- [matplotlib](http://matplotlib.org) 是一个用 Python 绘制图形的库。
- dnn_utils为此Notebook提供了一些必要的函数。
- testCases 提供了一些测试用例来评估你的函数的正确性
- np.random.seed(1) 用于保持所有随机函数调用的一致性。它有助于对你的工作进行评分。请不要改变种子！ 

In [ ]:
### v1.2

In [ ]:
import numpy as np
import h5py
import matplotlib.pyplot as plt
from testCases import *
from dnn_utils import sigmoid, sigmoid_backward, relu, relu_backward
from public_tests import *

import copy
%matplotlib inline
plt.rcParams['figure.figsize'] = (5.0, 4.0) # 设置绘图的默认大小
plt.rcParams['image.interpolation'] = 'nearest'
plt.rcParams['image.cmap'] = 'gray'

%load_ext autoreload
%autoreload 2

np.random.seed(1)

<a name='2'></a>
## 2 - 概览

为了构建神经网络，你将实现几个“辅助函数”。这些辅助函数将在接下来的作业中使用来构建一个两层神经网络和一个L层神经网络。 

每个小帮手函数都会有详细的说明，引导你完成必要的步骤。以下是此作业中的步骤概述：

- 初始化两层网络和 $L$ 层神经网络的参数
- 实现前向传播模块（下图中紫色部分）
     - 完成层前向传播步骤的线性部分（结果为 $Z^{[l]}$）。
     - 为你提供激活函数 (ReLU/sigmoid)
     - 将前两步组合成新的 [LINEAR->ACTIVATION] 前向传播函数。
     - 将 [LINEAR->RELU] 前向传播重复 L-1 次（对应第 1 至 L-1 层），并在最后添加 [LINEAR->SIGMOID]（对应第 $L$ 层），构成 L_model_forward 函数。
- 计算损失
- 实现反向传播模块（下图中红色表示）
    - 完成层反向传播步骤的线性部分
    - 激活函数的梯度是为 you(relu_backward/sigmoid_backward) 提供的 
    - 将前两步组合成新的 [LINEAR->ACTIVATION] 反向传播函数。
    - 在新的 L_model_backward 函数中，组合 [LINEAR->SIGMOID] 的反向传播与重复 L-1 次的 [LINEAR->RELU] 反向传播。
- 最后更新参数

<img src="images/final outline.png" style="width:800px;height:500px;">
<caption><center><b>图1</b></center></caption><br>


**注意**：

每个前向传播函数都有对应的反向传播函数。因此，前向传播的每一步都需要缓存一些中间值，以供反向传播计算梯度时使用。

在反向传播模块中，你可以使用缓存来计算梯度。别担心，这个作业将向你准确展示如何执行这些步骤！ 

<a name='3'></a>
## 3 - 初始化

你将编写两个辅助函数来初始化模型的参数。第一个函数将用于初始化两层模型的参数。第二个将这个初始化过程推广到 $L$ 层。

<a name='3-1'></a>
### 3.1 - 2 层神经网络

<a name='ex-1'></a>
### 练习 1 - initialize_parameters

创建并初始化 2 层神经网络的参数。

**说明**：

- 模型结构为：*LINEAR -> ReLU -> LINEAR -> sigmoid*。 
- 使用此随机初始化权重矩阵：`np.random.randn(d0, d1, ..., dn) * 0.01` 具有正确的形状。 [np.random.randn](https://numpy.org/doc/stable/reference/random/generated/numpy.random.randn.html) 的文档
- 对偏置使用零初始化：`np.zeros(shape)`。 [np.zeros](https://numpy.org/doc/stable/reference/generated/numpy.zeros.html) 的文档

In [ ]:
# GRADED FUNCTION: initialize_parameters

def initialize_parameters(n_x, n_h, n_y):
    """
    论证：
    n_x -- 输入层的大小
    n_h -- 隐藏层的尺寸
    n_y -- 输出层的大小
    
    返回：
    parameters -- 包含参数的Python字典：
                    W1 -- 形状 (n_h, n_x) 的权重矩阵
                    b1 -- 形状为 (n_h, 1) 的偏置向量
                    W2 -- 形状 (n_y, n_h) 的权重矩阵
                    b2 -- 形状为 (n_y, 1) 的偏置向量
    """
    
    np.random.seed(1)
    
    #（约4行代码）
    # W1 = ...
    # b1 = ...
    # W2 = ...
    # b2 = ...
    # YOUR CODE STARTS HERE
    
    
    # YOUR CODE ENDS HERE
    
    parameters = {"W1": W1,
                  "b1": b1,
                  "W2": W2,
                  "b2": b2}
    
    return parameters    

In [ ]:
print("Test Case 1:\n")
parameters = initialize_parameters(3,2,1)

print("W1 = " + str(parameters["W1"]))
print("b1 = " + str(parameters["b1"]))
print("W2 = " + str(parameters["W2"]))
print("b2 = " + str(parameters["b2"]))

initialize_parameters_test_1(initialize_parameters)

print("\033[90m\nTest Case 2:\n")
parameters = initialize_parameters(4,3,2)

print("W1 = " + str(parameters["W1"]))
print("b1 = " + str(parameters["b1"]))
print("W2 = " + str(parameters["W2"]))
print("b2 = " + str(parameters["b2"]))

initialize_parameters_test_2(initialize_parameters)

***预期输出***
```
Test Case 1:

W1 = [[ 0.01624345 -0.00611756 -0.00528172]
 [-0.01072969  0.00865408 -0.02301539]]
b1 = [[0.]
 [0.]]
W2 = [[ 0.01744812 -0.00761207]]
b2 = [[0.]]
 All tests passed.

Test Case 2:

W1 = [[ 0.01624345 -0.00611756 -0.00528172 -0.01072969]
 [ 0.00865408 -0.02301539  0.01744812 -0.00761207]
 [ 0.00319039 -0.0024937   0.01462108 -0.02060141]]
b1 = [[0.]
 [0.]
 [0.]]
W2 = [[-0.00322417 -0.00384054  0.01133769]
 [-0.01099891 -0.00172428 -0.00877858]]
b2 = [[0.]
 [0.]]
 All tests passed.
```

<a name='3-2'></a>
### 3.2 - L层神经网络

更深的L层神经网络的初始化更加复杂，因为有更多的权重矩阵和偏置向量。完成 `initialize_parameters_deep` 函数时，你应该确保每层之间的尺寸匹配。回想一下，$n^{[l]}$ 是 $l$ 层中的单元数。例如，如果输入 $X$ 的大小为 $(12288, 209)$（以 $m=209$ 示例），则：

<table style="width:100%">
    <tr>
        <td>  </td> 
        <td> <b>W</b> </td> 的形状 
        <td> <b> b</b> </td> 的形状 
        <td> <b>激活</b> </td>
        <td> <b>激活形状</b> </td> 
    <tr>
    <tr>
        <td> <b>第1层</b> </td> 
        <td> $(n^{[1]},12288)$ </td> 
        <td> $(n^{[1]},1)$ </td> 
        <td> $Z^{[1]} = W^{[1]}  X + b^{[1]} $ </td> 
        <td> $(n^{[1]},209)$ </td> 
    <tr>
    <tr>
        <td> <b>第2层</b> </td> 
        <td> $(n^{[2]}, n^{[1]})$  </td> 
        <td> $(n^{[2]},1)$ </td> 
        <td>$Z^{[2]} = W^{[2]} A^{[1]} + b^{[2]}$ </td> 
        <td> $(n^{[2]}, 209)$ </td> 
    <tr>
       <tr>
        <td> $\vdots$ </td> 
        <td> $\vdots$  </td> 
        <td> $\vdots$  </td> 
        <td> $\vdots$</td> 
        <td> $\vdots$  </td> 
    <tr>  
   <tr>
       <td> <b>L-1</b> </td> 
        <td> $(n^{[L-1]}, n^{[L-2]})$ </td> 
        <td> $(n^{[L-1]}, 1)$  </td> 
        <td>$Z^{[L-1]} =  W^{[L-1]} A^{[L-2]} + b^{[L-1]}$ </td> 
        <td> $(n^{[L-1]}, 209)$ </td> 
   <tr>
   <tr>
       <td> <b>层 L</b> </td> 
        <td> $(n^{[L]}, n^{[L-1]})$ </td> 
        <td> $(n^{[L]}, 1)$ </td>
        <td> $Z^{[L]} =  W^{[L]} A^{[L-1]} + b^{[L]}$</td>
        <td> $(n^{[L]}, 209)$  </td> 
    <tr>
</table>

请记住，当你在 python 中计算 $W X + b$ 时，它会执行广播。例如，如果： 

$$ W = \begin{bmatrix}
    w_{00}  & w_{01} & w_{02} \\
    w_{10}  & w_{11} & w_{12} \\
    w_{20}  & w_{21} & w_{22} 
\end{bmatrix}\;\;\; X = \begin{bmatrix}
    x_{00}  & x_{01} & x_{02} \\
    x_{10}  & x_{11} & x_{12} \\
    x_{20}  & x_{21} & x_{22} 
\end{bmatrix} \;\;\; b =\begin{bmatrix}
    b_0  \\
    b_1  \\
    b_2
\end{bmatrix}\tag{2}$$

那么 $WX + b$ 将是：

$$ WX + b = \begin{bmatrix}
    (w_{00}x_{00} + w_{01}x_{10} + w_{02}x_{20}) + b_0 & (w_{00}x_{01} + w_{01}x_{11} + w_{02}x_{21}) + b_0 & \cdots \\
    (w_{10}x_{00} + w_{11}x_{10} + w_{12}x_{20}) + b_1 & (w_{10}x_{01} + w_{11}x_{11} + w_{12}x_{21}) + b_1 & \cdots \\
    (w_{20}x_{00} + w_{21}x_{10} + w_{22}x_{20}) + b_2 &  (w_{20}x_{01} + w_{21}x_{11} + w_{22}x_{21}) + b_2 & \cdots
\end{bmatrix}\tag{3}  $$


<a name='ex-2'></a>
### 练习 2 - initialize_parameters_deep

实现 L 层神经网络的初始化。 

**说明**：
- 模型结构为 *[LINEAR -> RELU] $ \times$ (L-1) -> LINEAR -> SIGMOID* 。即包含 $L-1$ 个使用 ReLU 激活函数的隐藏层，最后连接一个使用 sigmoid 激活函数的输出层。
- 对权重矩阵使用随机初始化。使用 `np.random.randn(d0, d1, ..., dn) * 0.01`。
- 对偏置使用零初始化。使用 `np.zeros(shape)`。
- 你将在变量 `layer_dims` 中存储 $n^{[l]}$（不同层中的单元数）。例如，上周平面数据分类模型的 `layer_dims` 将为 [2,4,1]：有两个输入，一个具有 4 个隐藏单元的隐藏层，一个具有 1 个输出单元的输出层。这意味着`W1`的形状是(4,2)，`b1`是(4,1)，`W2`是(1,4)，`b2`是(1,1)。现在你将把它推广到 $L$ 层！ 
- 这是 $L=1$（一层神经网络）的实现。它应该会启发你实现一般情况（L 层神经网络）。
```python
    if L == 1:
        parameters["W" + str(L)] = np.random.randn(layer_dims[1], layer_dims[0]) * 0.01
        parameters["b" + str(L)] = np.zeros((layer_dims[1], 1))
```

In [ ]:
# GRADED FUNCTION: initialize_parameters_deep

def initialize_parameters_deep(layer_dims):
    """
    参数：
    layer_dims -- 包含网络中每层维度的 python 数组（列表）
    
    返回：
    parameters -- 包含参数 "W1"、"b1"、...、"WL"、"bL" 的 python 字典：
                    Wl -- 形状权重矩阵 (layer_dims[l], layer_dims[l-1])
                    bl -- 形状的偏置向量 (layer_dims[l], 1)
    """
    
    np.random.seed(3)
    parameters = {}
    L = len(layer_dims) # 网络层数

    for l in range(1, L):
        #（约2行代码）
        # parameters['W' + str(l)] = ...
        # parameters['b' + str(l)] = ...
        # YOUR CODE STARTS HERE
        
        
        # YOUR CODE ENDS HERE
        
        assert(parameters['W' + str(l)].shape == (layer_dims[l], layer_dims[l - 1]))
        assert(parameters['b' + str(l)].shape == (layer_dims[l], 1))

        
    return parameters

In [ ]:
print("Test Case 1:\n")
parameters = initialize_parameters_deep([5,4,3])

print("W1 = " + str(parameters["W1"]))
print("b1 = " + str(parameters["b1"]))
print("W2 = " + str(parameters["W2"]))
print("b2 = " + str(parameters["b2"]))

initialize_parameters_deep_test_1(initialize_parameters_deep)

print("\033[90m\nTest Case 2:\n")
parameters = initialize_parameters_deep([4,3,2])

print("W1 = " + str(parameters["W1"]))
print("b1 = " + str(parameters["b1"]))
print("W2 = " + str(parameters["W2"]))
print("b2 = " + str(parameters["b2"]))
initialize_parameters_deep_test_2(initialize_parameters_deep)

***预期输出***
```
Test Case 1:

W1 = [[ 0.01788628  0.0043651   0.00096497 -0.01863493 -0.00277388]
 [-0.00354759 -0.00082741 -0.00627001 -0.00043818 -0.00477218]
 [-0.01313865  0.00884622  0.00881318  0.01709573  0.00050034]
 [-0.00404677 -0.0054536  -0.01546477  0.00982367 -0.01101068]]
b1 = [[0.]
 [0.]
 [0.]
 [0.]]
W2 = [[-0.01185047 -0.0020565   0.01486148  0.00236716]
 [-0.01023785 -0.00712993  0.00625245 -0.00160513]
 [-0.00768836 -0.00230031  0.00745056  0.01976111]]
b2 = [[0.]
 [0.]
 [0.]]
 All tests passed.

Test Case 2:

W1 = [[ 0.01788628  0.0043651   0.00096497 -0.01863493]
 [-0.00277388 -0.00354759 -0.00082741 -0.00627001]
 [-0.00043818 -0.00477218 -0.01313865  0.00884622]]
b1 = [[0.]
 [0.]
 [0.]]
W2 = [[ 0.00881318  0.01709573  0.00050034]
 [-0.00404677 -0.0054536  -0.01546477]]
b2 = [[0.]
 [0.]]
 All tests passed.
```

<a name='4'></a>
## 4 - 前向传播模块

<a name='4-1'></a>
### 4.1 - 线性部分的前向传播 

现在你已经初始化了参数，你可以做前向传播模块了。首先实现一些基本的函数，你可以在稍后实现模型时再次使用它们。现在，你将按以下顺序完成三个函数：

- 线性
- 线性 -> 激活值，其中激活值将为 ReLU 或 sigmoid。 
- [线性 -> ReLU] $\times$ (L-1) -> 线性 -> sigmoid（整个模型）

线性正向模块（所有示例中的向量化的）计算以下方程：

$$Z^{[l]} = W^{[l]}A^{[l-1]} +b^{[l]}\tag{4}$$

其中 $A^{[0]} = X$。 

<a name='ex-3'></a>
### 练习 3 - linear_forward 

构建前向传播的线性部分。

**提醒**：
该单位的数学表示为$Z^{[l]} = W^{[l]}A^{[l-1]} +b^{[l]}$。你可能还会发现 `np.dot()` 很有用。如果你的尺寸不匹配，打印 `W.shape` 可能会有所帮助。

In [ ]:
# GRADED FUNCTION: linear_forward

def linear_forward(A, W, b):
    """
    实现层前向传播的线性部分。

    参数：
    A -- 来自上一层（或输入数据）的激活值：（上一层的大小，示例数）
    W -- 权重矩阵：NumPy 形状数组（当前层的大小，前一层的大小）
    b -- 偏置向量，NumPy 形状数组（当前层的大小，1）

    返回：
    Z -- 激活函数的输入，也称为前置激活值参数 
    cache -- 包含 "A"、"W" 和 "b" 的 Python 元组；存储用于有效计算反向传播
    """
    
    #（≈ 1 行代码）
    # Z = ...
    # YOUR CODE STARTS HERE
    
    
    # YOUR CODE ENDS HERE
    cache = (A, W, b)
    
    return Z, cache

In [ ]:
t_A, t_W, t_b = linear_forward_test_case()
t_Z, t_linear_cache = linear_forward(t_A, t_W, t_b)
print("Z = " + str(t_Z))

linear_forward_test(linear_forward)

***预期输出***
```
Z = [[ 3.26295337 -1.23429987]]
```

<a name='4-2'></a>
### 4.2 - 线性与激活部分的前向传播

在这个Notebook中，你将使用两个激活函数：

- **sigmoid**：$\sigma(Z) = \sigma(W A + b) = \frac{1}{ 1 + e^{-(W A + b)}}$。你已获得 `sigmoid` 函数，它返回**两个**项目：激活值“`a`”和包含“`Z`”的“`cache`”（这是我们将输入到相应的反向传播函数）。要使用它，你只需调用： 
``` python
A, activation_cache = sigmoid(Z)
```

- **ReLU**：ReLU 的数学公式为 $A = RELU(Z) = max(0, Z)$。你已获得 `relu` 函数。此函数返回**两个**项目：激活值“`A`”和包含“`Z`”的“`cache`”（这是你将输入到相应的反向传播函数的内容）。要使用它，你只需调用：
``` python
A, activation_cache = relu(Z)
```

为了更加方便，你将把两个函数（线性和激活值）分组为一个函数（线性 -> 激活值）。因此，你将实现一个执行线性前向步骤的函数，然后执行激活值前向步骤。

<a name='ex-4'></a>
### 练习 4 - linear_activation_forward

实现 *LINEAR->激活值* 层的前向传播。数学关系为：$A^{[l]} = g(Z^{[l]}) = g(W^{[l]}A^{[l-1]} +b^{[l]})$，其中激活值“g”可以是sigmoid()或relu()。使用 `linear_forward()` 和正确的激活函数。

In [ ]:
# GRADED FUNCTION: linear_activation_forward

def linear_activation_forward(A_prev, W, b, activation):
    """
    为 LINEAR->激活层实现前向传播

    参数：
    A_prev -- 来自上一层（或输入数据）的激活值：（上一层的大小，示例数量）
    W -- 权重矩阵：NumPy 形状数组（当前层的大小，前一层的大小）
    b -- 偏置向量，NumPy 形状数组（当前层的大小，1）
    activation -- 该层要使用的激活值，存储为文本字符串："sigmoid" 或 "relu"

    返回：
    A -- 激活函数的输出，也称为激活输出 
    cache -- 包含 "linear_cache" 和 "activation_cache" 的 Python 元组； 存储
             以有效计算反向传播
    """
    
    if activation == "sigmoid":
        #（约2行代码）
        # Z, linear_cache = ...
        # A, activation_cache = ...
        # YOUR CODE STARTS HERE
        
        
        # YOUR CODE ENDS HERE
    
    elif activation == "relu":
        #（约2行代码）
        # Z, linear_cache = ...
        # A, activation_cache = ...
        # YOUR CODE STARTS HERE
        
        
        # YOUR CODE ENDS HERE
    cache = (linear_cache, activation_cache)

    return A, cache

In [ ]:
t_A_prev, t_W, t_b = linear_activation_forward_test_case()

t_A, t_linear_activation_cache = linear_activation_forward(t_A_prev, t_W, t_b, activation = "sigmoid")
print("With sigmoid: A = " + str(t_A))

t_A, t_linear_activation_cache = linear_activation_forward(t_A_prev, t_W, t_b, activation = "relu")
print("With ReLU: A = " + str(t_A))

linear_activation_forward_test(linear_activation_forward)

***预期输出***
```
With sigmoid: A = [[0.96890023 0.11013289]]
With ReLU: A = [[3.43896131 0.        ]]
```

**注意**：在深度学习中，“[LINEAR->激活值]”计算被视为神经网络中的单层，而不是两层。 

<a name='4-3'></a>
### 4.3 - L层模型 

为了让 $L$ 层神经网络的实现*更加*方便，需要编写一个函数：将使用 ReLU 的 `linear_activation_forward` 重复调用 $L-1$ 次，最后再调用一次使用 sigmoid 的 `linear_activation_forward`。

<img src="images/model_architecture_kiank.png" style="width:600px;height:300px;">
<caption><center> <b>图 2</b> : *[线性 -> ReLU] $\times$ (L-1) -> 线性 -> sigmoid*模型</center></caption><br>

<a name='ex-5'></a>
### 练习 5 - L_model_forward

实现上述模型的前向传播。

**说明**：在下面的代码中，变量`AL`将表示$A^{[L]} = \sigma(Z^{[L]}) = \sigma(W^{[L]} A^{[L-1]} + b^{[L]})$。 （有时也称为 `Yhat`、即这是 $\hat{Y}$。） 

**提示**：
- 使用你之前编写的函数 
- 使用for循环复制[LINEAR->ReLU] (L-1)次
- 不要忘记跟踪“缓存”列表中的缓存。要将新值 `c` 添加到 `list`，你可以使用 `list.append(c)`。

In [ ]:
# GRADED FUNCTION: L_model_forward

def L_model_forward(X, parameters):
    """
    为 [LINEAR->ReLU]*(L-1)->LINEAR->sigmoid 计算实现前向传播
    
    参数：
    X -- 数据，NumPy 形状数组（输入大小，示例数量）
    parameters -- initialize_parameters_deep()的输出
    
    返回：
    AL -- 来自输出（最后）层的激活值
    caches -- 缓存列表包含：
                linear_activation_forward()的每个缓存（有L个，索引从0到L-1）
    """

    caches = []
    A = X
    L = len(parameters) // 2                  # 神经网络的层数
    
    # 实现 [线性 -> ReLU]*(L-1)。将 "cache" 添加到 "caches" 列表。
    # for 循环从 1 开始，因为第 0 层是输入
    for l in range(1, L):
        A_prev = A 
        #（约2行代码）
        # A, cache = ...
        # 缓存...
        # YOUR CODE STARTS HERE
        
        
        # YOUR CODE ENDS HERE
    
    # 实现线性 -> sigmoid。将 "cache" 添加到 "caches" 列表。
    #（约2行代码）
    # AL, cache = ...
    # 缓存...
    # YOUR CODE STARTS HERE
    
    
    # YOUR CODE ENDS HERE
          
    return AL, caches

In [ ]:
t_X, t_parameters = L_model_forward_test_case_2hidden()
t_AL, t_caches = L_model_forward(t_X, t_parameters)

print("AL = " + str(t_AL))

L_model_forward_test(L_model_forward)

***预期输出***
```
AL = [[0.03921668 0.70498921 0.19734387 0.04728177]]
```

**太棒了！** 你已经实现了完整的前向传播，它接受输入 X 并输出包含你的预测的行向量 $A^{[L]}$。它还将所有中间值记录在“缓存”中。使用 $A^{[L]}$，你可以计算预测的代价。

<a name='5'></a>
## 5 - 代价函数

现在你可以实现forward和反向传播了！你需要计算代价，以检查你的模型是否真正在学习。

<a name='ex-6'></a>
### 练习 6 - compute_cost
使用以下公式计算交叉熵代价 $J$：$$-\frac{1}{m} \sum\limits_{i = 1}^{m} (y^{(i)}\log\left(a^{[L] (i)}\right) + (1-y^{(i)})\log\left(1- a^{[L](i)}\right)) \tag{7}$$


In [ ]:
# GRADED FUNCTION: compute_cost

def compute_cost(AL, Y):
    """
    实现等式(7)定义的代价函数。

    参数：
    AL -- 对应于标签预测的概率向量，形状（1，示例数量）
    Y -- true "label" 向量（例如：如果非猫则包含 0，如果猫则包含 1），形状（1，示例数量）

    返回：
    cost -- 交叉熵代价
    """
    
    m = Y.shape[1]

    # 计算 aL 和 y 的损失。
    # （约1行代码）
    # cost = ...
    # YOUR CODE STARTS HERE
    
    
    # YOUR CODE ENDS HERE
    
    cost = np.squeeze(cost)      # 确保你的代价形状符合我们的预期（例如这会将 [[17]] 变成 17）。

    
    return cost

In [ ]:
t_Y, t_AL = compute_cost_test_case()
t_cost = compute_cost(t_AL, t_Y)

print("Cost: " + str(t_cost))

compute_cost_test(compute_cost)

**预期输出**：

<table>
    <tr>
        <td><b>代价</b> </td>
    <td> 0.2797765635793422</td> 
    </tr>
</table>

<a name='6'></a>
## 6 - 反向传播模块

正如你对前向传播所做的那样，你将为反向传播实现辅助函数。请记住，反向传播用于计算损失函数相对于参数的梯度。 

**提醒**： 
<img src="images/backprop_kiank.png" style="width:650px;height:250px;">
<caption><center><font color='purple'><b>图 3</b>：LINEAR->RELU->LINEAR->SIGMOID 的前向传播与反向传播<br> <i>紫色模块表示前向传播，红色模块表示反向传播。</font></center></caption>


<!-- 
如果你熟悉微积分（完成本作业并不要求精通微积分），可以用链式法则推导两层网络中损失 $\mathcal{L}$ 相对于 $z^{[1]}$ 的导数：

$$\frac{d \mathcal{L}(a^{[2]},y)}{{dz^{[1]}}} = \frac{d\mathcal{L}(a^{[2]},y)}{{da^{[2]}}}\frac{{da^{[2]}}}{{dz^{[2]}}}\frac{{dz^{[2]}}}{{da^{[1]}}}\frac{{da^{[1]}}}{{dz^{[1]}}} \tag{8} $$

为了计算梯度 $dW^{[1]} = \frac{\partial L}{\partial W^{[1]}}$，请使用前面的链式法则，然后执行 $dW^{[1]} = dz^{[1]} \times \frac{\partial z^{[1]} }{\partial W^{[1]}}$。在反向传播期间，在每一步中，你都将当前的梯度乘以与特定层对应的梯度，以获得你想要的梯度。

同样，为了计算梯度 $db^{[1]} = \frac{\partial L}{\partial b^{[1]}}$，你可以使用前面的链式法则并执行 $db^{[1]} = dz^{[1]} \times \frac{\partial z^{[1]} }{\partial b^{[1]}}$。

这就是我们谈论 **反向传播** 的原因。
!-->

现在，与前向传播类似，你将分三个步骤构建反向传播：
1. 线性部分的反向传播
2. LINEAR -> ACTIVATION 的反向传播，其中 ACTIVATION 部分计算 ReLU 或 sigmoid 激活函数的导数。
3. [LINEAR -> RELU] $\times$ (L-1) -> LINEAR -> SIGMOID 的反向传播（整个模型）。

对于下一个练习，你需要记住：

- `b` 是一个 1 列 n 行的 matrix(np.ndarray)，i.e：b = [[1.0], [2.0]] （记住 `b` 是一个常数）
- np.sum 对 ndarray 的元素执行求和
- axis=1 或 axis=0 分别指定是按行求和还是按列求和
- keepdims 指定是否必须保留矩阵的原始尺寸。
- 看下面的例子来澄清：

In [ ]:
A = np.array([[1, 2], [3, 4]])

print('axis=1 and keepdims=True')
print(np.sum(A, axis=1, keepdims=True))
print('axis=1 and keepdims=False')
print(np.sum(A, axis=1, keepdims=False))
print('axis=0 and keepdims=True')
print(np.sum(A, axis=0, keepdims=True))
print('axis=0 and keepdims=False')
print(np.sum(A, axis=0, keepdims=False))

<a name='6-1'></a>
### 6.1 - 线性部分的反向传播

对于层 $l$，线性部分为：$Z^{[l]} = W^{[l]} A^{[l-1]} + b^{[l]}$（后跟激活值）。

假设你已经计算了导数 $dZ^{[l]} = \frac{\partial \mathcal{L} }{\partial Z^{[l]}}$。你想要获得 $(dW^{[l]}, db^{[l]}, dA^{[l-1]})$。

<img src="images/linearback_kiank.png" style="width:250px;height:300px;">
<caption><center><font color='purple'><b>图4</b></font></center></caption>

三个输出 $(dW^{[l]}, db^{[l]}, dA^{[l-1]})$ 使用输入 $dZ^{[l]}$ 进行计算。

以下是你需要的公式：
$$ dW^{[l]} = \frac{\partial \mathcal{J} }{\partial W^{[l]}} = \frac{1}{m} dZ^{[l]} A^{[l-1] T} \tag{8}$$
$$ db^{[l]} = \frac{\partial \mathcal{J} }{\partial b^{[l]}} = \frac{1}{m} \sum_{i = 1}^{m} dZ^{[l](i)}\tag{9}$$
$$ dA^{[l-1]} = \frac{\partial \mathcal{L} }{\partial A^{[l-1]}} = W^{[l] T} dZ^{[l]} \tag{10}$$


$A^{[l-1] T}$ 是 $A^{[l-1]}$ 的转置。 

<a name='ex-7'></a>
### 练习 7 - linear_backward 

使用上面的3个公式来实现`linear_backward()`。

**提示**：

- 在 NumPy 中，你可以使用 `A.T` 或 `A.transpose()` 获得 ndarray `A` 的转置

In [ ]:
# GRADED FUNCTION: linear_backward

def linear_backward(dZ, cache):
    """
    实现单层（l 层）反向传播的线性部分

    参数：
    dZ -- 代价相对于当前第 l 层线性输出的梯度
    cache -- 来自当前层中的前向传播的值 (A_prev, W, b) 的元组

    返回：
    dA_prev -- 代价相对于前一层 l-1 的激活值的梯度，形状与 A_prev 相同
    dW -- 代价相对于当前第 l 层权重 W 的梯度，形状与 W 相同
    db -- 代价相对于当前第 l 层偏置 b 的梯度，形状与 b 相同
    """
    A_prev, W, b = cache
    m = A_prev.shape[1]

    ### START CODE HERE ### (≈ 3 lines of code)
    # dW = ...
    # db = ... 通过 keepdims=True 的 dZ 行求和
    # dA_prev = ...
    # YOUR CODE STARTS HERE
    
    
    # YOUR CODE ENDS HERE
    
    return dA_prev, dW, db

In [ ]:
t_dZ, t_linear_cache = linear_backward_test_case()
t_dA_prev, t_dW, t_db = linear_backward(t_dZ, t_linear_cache)

print("dA_prev: " + str(t_dA_prev))
print("dW: " + str(t_dW))
print("db: " + str(t_db))

linear_backward_test(linear_backward)

**预期输出**：
```
dA_prev: [[-1.15171336  0.06718465 -0.3204696   2.09812712]
 [ 0.60345879 -3.72508701  5.81700741 -3.84326836]
 [-0.4319552  -1.30987417  1.72354705  0.05070578]
 [-0.38981415  0.60811244 -1.25938424  1.47191593]
 [-2.52214926  2.67882552 -0.67947465  1.48119548]]
dW: [[ 0.07313866 -0.0976715  -0.87585828  0.73763362  0.00785716]
 [ 0.85508818  0.37530413 -0.59912655  0.71278189 -0.58931808]
 [ 0.97913304 -0.24376494 -0.08839671  0.55151192 -0.10290907]]
db: [[-0.14713786]
 [-0.11313155]
 [-0.13209101]]
 ```

<a name='6-2'></a>
### 6.2 - 线性与激活部分的反向传播

接下来，创建 **`linear_activation_backward`** 函数，将 **`linear_backward`** 与激活函数的反向传播步骤组合起来。

为了帮助你实现`linear_activation_backward`，提供了两个反向传播函数：
- **`sigmoid_backward`**：为 sigmoid 单元实现反向传播。你可以按如下方式调用它：

```python
dZ = sigmoid_backward(dA, activation_cache)
```

- **`relu_backward`**：为 ReLU 单元实现反向传播。你可以按如下方式调用它：

```python
dZ = relu_backward(dA, activation_cache)
```

如果 $g(.)$ 是激活函数， 
`sigmoid_backward` 和 `relu_backward` 计算 $$dZ^{[l]} = dA^{[l]} * g'(Z^{[l]}). \tag{11}$$  

<a name='ex-8'></a>
### 练习 8 - linear_activation_backward

为 *LINEAR->激活值* 层实现反向传播。

In [ ]:
# GRADED FUNCTION: linear_activation_backward

def linear_activation_backward(dA, cache, activation):
    """
    为 LINEAR->激活层实现反向传播。
    
    参数：
    dA -- 当前层 l 的激活输出梯度 
    cache -- 我们存储的值 (linear_cache, activation_cache) 的元组用于有效计算反向传播
    activation -- 该层要使用的激活值，存储为文本字符串："sigmoid" 或 "relu"
    
    返回：
    dA_prev -- 梯度相对于激活值（前一层 l-1）的代价，与 A_prev 形状相同
    dW -- 梯度相对于 W（当前层 l）的代价，与 W 形状相同
    db -- 梯度相对于 b（当前层 l）的代价，与 b 形状相同
    """
    linear_cache, activation_cache = cache
    
    if activation == "relu":
        #（约2行代码）
        # dZ =  ...
        # dA_prev, dW, db =  ...
        # YOUR CODE STARTS HERE
        
        
        # YOUR CODE ENDS HERE
        
    elif activation == "sigmoid":
        #（约2行代码）
        # dZ =  ...
        # dA_prev, dW, db =  ...
        # YOUR CODE STARTS HERE
        
        
        # YOUR CODE ENDS HERE
    
    return dA_prev, dW, db

In [ ]:
t_dAL, t_linear_activation_cache = linear_activation_backward_test_case()

t_dA_prev, t_dW, t_db = linear_activation_backward(t_dAL, t_linear_activation_cache, activation = "sigmoid")
print("With sigmoid: dA_prev = " + str(t_dA_prev))
print("With sigmoid: dW = " + str(t_dW))
print("With sigmoid: db = " + str(t_db))

t_dA_prev, t_dW, t_db = linear_activation_backward(t_dAL, t_linear_activation_cache, activation = "relu")
print("With relu: dA_prev = " + str(t_dA_prev))
print("With relu: dW = " + str(t_dW))
print("With relu: db = " + str(t_db))

linear_activation_backward_test(linear_activation_backward)

**预期输出：**

```
With sigmoid: dA_prev = [[ 0.11017994  0.01105339]
 [ 0.09466817  0.00949723]
 [-0.05743092 -0.00576154]]
With sigmoid: dW = [[ 0.10266786  0.09778551 -0.01968084]]
With sigmoid: db = [[-0.05729622]]
With relu: dA_prev = [[ 0.44090989  0.        ]
 [ 0.37883606  0.        ]
 [-0.2298228   0.        ]]
With relu: dW = [[ 0.44513824  0.37371418 -0.10478989]]
With relu: db = [[-0.20837892]]
```

<a name='6-3'></a>
### 6.3 - L 层模型的反向传播 

现在，实现整个网络的反向传播函数！

回想一下，当你实现 `L_model_forward` 函数时，在每次迭代时，你都存储了一个包含（X、W、b 和 z）的缓存。在反向传播模块中，你将使用这些变量来计算梯度。因此，在 `L_model_backward` 函数中，你将从 $L$ 层开始向后迭代所有隐藏层。在每个步骤中，你将使用层 $l$ 的缓存值通过层 $l$ 进行反向传播。下图 5 显示了反向传播。 


<img src="images/mn_backward.png" style="width:450px;height:300px;">
<caption><center><font color='purple'><b>图5</b>：反向传播</font></center></caption>

**初始化反向传播**：

要通过该网络进行反向传播，你知道输出是： 
$A^{[L]} = \sigma(Z^{[L]})$。因此，你的代码需要计算 `dAL` $= \frac{\partial \mathcal{L}}{\partial A^{[L]}}$。
为此，请使用以下公式（使用微积分导出，同样，你不需要深入了解！）：
```python
dAL = - (np.divide(Y, AL) - np.divide(1 - Y, 1 - AL)) # 代价相对于 AL 的导数
```

接下来，利用代价对激活输出的梯度 `dAL` 继续反向传播。如图 5 所示，将 `dAL` 传入已实现的 LINEAR->SIGMOID 反向传播函数，该函数会使用 L_model_forward 缓存的中间值。

之后，用 `for` 循环遍历其余各层，调用 LINEAR->RELU 的反向传播函数，并将各层的 dA、dW 和 db 存入 grads 字典。具体使用以下公式：

$$grads["dW" + str(l)] = dW^{[l]}\tag{15} $$

例如，对于 $l=3$，这会将 $dW^{[l]}$ 存储在 `grads["dW3"]` 中。

<a name='ex-9'></a>
### 练习 9 - L_model_backward

为 *[LINEAR->ReLU] $\times$ (L-1) -> LINEAR -> sigmoid* 模型实现反向传播。

In [ ]:
# GRADED FUNCTION: L_model_backward

def L_model_backward(AL, Y, caches):
    """
    为 [LINEAR->ReLU] * (L-1) -> LINEAR -> sigmoid 组实现反向传播
    
    参数：
    AL -- 概率向量，前向传播 (L_model_forward()) 的输出
    Y -- true "label" 向量（如果非猫则包含 0，如果是猫则包含 1）
    caches -- 缓存列表包含：
                linear_activation_forward() 和 "relu" 的每个缓存（它是缓存[l]，对于 range(L-1) i.e l = 0...L-2 中的 l）
                linear_activation_forward() 与 "sigmoid" 的缓存（它是缓存[L-1]）
    
    返回：
    grads -- 包含各层梯度的字典
             grads["dA" + str(l)] = ... 
             grads["dW" + str(l)] = ...
             grads["db" + str(l)] = ... 
    """
    grads = {}
    L = len(caches) # 层数
    m = AL.shape[1]
    Y = Y.reshape(AL.shape) # 在此行之后，Y 与 AL 形状相同
    
    # 初始化反向传播
    #（1行代码）
    # dAL = ...
    # YOUR CODE STARTS HERE
    
    
    # YOUR CODE ENDS HERE
    
    # 第 L 层（sigmoid -> 线性）梯度。输入："dAL, current_cache"。输出：“梯度["dAL-1"]，梯度["dWL"]，梯度["dbL"]
    #（约5行）
    # current_cache = ...
    # dA_prev_temp, dW_temp, db_temp = ...
    # grads["dA" + str(L-1)] = ...
    # grads["dW" + str(L)] = ...
    # grads["db" + str(L)] = ...
    # YOUR CODE STARTS HERE
    
    
    # YOUR CODE ENDS HERE
    
    # 从 l=L-2 循环到 l=0
    for l in reversed(range(L-1)):
        # 第 l 层：（ReLU -> 线性）梯度。
        # 输入：“grads["dA" + str(l + 1)]，current_cache”。输出：“梯度["dA" + str(l)]，梯度["dW" + str(l + 1)]，梯度["db" + str(l + 1)] 
        #（约5行）
        # current_cache = ...
        # dA_prev_temp, dW_temp, db_temp = ...
        # grads["dA" + str(l)] = ...
        # grads["dW" + str(l + 1)] = ...
        # grads["db" + str(l + 1)] = ...
        # YOUR CODE STARTS HERE
        
        
        # YOUR CODE ENDS HERE

    return grads

In [ ]:
t_AL, t_Y_assess, t_caches = L_model_backward_test_case()
grads = L_model_backward(t_AL, t_Y_assess, t_caches)

print("dA0 = " + str(grads['dA0']))
print("dA1 = " + str(grads['dA1']))
print("dW1 = " + str(grads['dW1']))
print("dW2 = " + str(grads['dW2']))
print("db1 = " + str(grads['db1']))
print("db2 = " + str(grads['db2']))

L_model_backward_test(L_model_backward)

**预期输出：**

```
dA0 = [[ 0.          0.52257901]
 [ 0.         -0.3269206 ]
 [ 0.         -0.32070404]
 [ 0.         -0.74079187]]
dA1 = [[ 0.12913162 -0.44014127]
 [-0.14175655  0.48317296]
 [ 0.01663708 -0.05670698]]
dW1 = [[0.41010002 0.07807203 0.13798444 0.10502167]
 [0.         0.         0.         0.        ]
 [0.05283652 0.01005865 0.01777766 0.0135308 ]]
dW2 = [[-0.39202432 -0.13325855 -0.04601089]]
db1 = [[-0.22007063]
 [ 0.        ]
 [-0.02835349]]
db2 = [[0.15187861]]
```

<a name='6-4'></a>
### 6.4 - 更新参数

在本部分中，你将使用梯度下降更新模型的参数： 

$$ W^{[l]} = W^{[l]} - \alpha \text{ } dW^{[l]} \tag{16}$$
$$ b^{[l]} = b^{[l]} - \alpha \text{ } db^{[l]} \tag{17}$$

，其中 $\alpha$ 是学习率。 

计算更新后的参数后，将其存储在参数字典中。 

<a name='ex-10'></a>
### 练习 10 - update_parameters

实现 `update_parameters()` 以使用梯度下降更新参数。

**说明**：
在每个 $W^{[l]}$ 和 $b^{[l]}$ 上使用梯度下降更新 $l = 1, 2, ..., L$ 的参数。 

In [ ]:
# GRADED FUNCTION: update_parameters

def update_parameters(params, grads, learning_rate):
    """
    使用梯度下降更新参数
    
    参数：
    params -- 包含参数的 python 字典 
    grads -- 包含梯度的 python 字典，L_model_backward 的输出
    
    返回：
    parameters -- 包含更新参数的Python字典 
                  parameters["W" + str(l)] = ... 
                  parameters["b" + str(l)] = ...
    """
    parameters = copy.deepcopy(params)
    L = len(parameters) // 2 # 神经网络的层数

    # 更新每个参数的规则。使用 for 循环。
    #（约2行代码）
    for l in range(L):
        # parameters["W" + str(l+1)] = ...
        # parameters["b" + str(l+1)] = ...
        # YOUR CODE STARTS HERE
        
        
        # YOUR CODE ENDS HERE
    return parameters

In [ ]:
t_parameters, grads = update_parameters_test_case()
t_parameters = update_parameters(t_parameters, grads, 0.1)

print ("W1 = "+ str(t_parameters["W1"]))
print ("b1 = "+ str(t_parameters["b1"]))
print ("W2 = "+ str(t_parameters["W2"]))
print ("b2 = "+ str(t_parameters["b2"]))

update_parameters_test(update_parameters)

**预期输出：**

```
W1 = [[-0.59562069 -0.09991781 -2.14584584  1.82662008]
 [-1.76569676 -0.80627147  0.51115557 -1.18258802]
 [-1.0535704  -0.86128581  0.68284052  2.20374577]]
b1 = [[-0.04659241]
 [-1.28888275]
 [ 0.53405496]]
W2 = [[-0.55569196  0.0354055   1.32964895]]
b2 = [[-0.84610769]]
```

### 恭喜！ 

你刚刚实现了构建深度神经网络所需的所有函数，包括： 

- 使用非线性单元改进你的模型
- 构建更深的神经网络（超过1个隐藏层）
- 实现一个易于使用的神经网络类

这确实是一个很长的作业，但是作业的下一部分更容易。 ;) 

在下一个作业中，你将把所有这些放在一起来构建两个模型：

- 两层神经网络
- L 层神经网络

事实上，你将使用这些模型来对猫与非猫图像进行分类！ （喵！）干得好，下次见。 